## 1. Import

In [185]:
import pandas as pd
import numpy as np
import json
import re
from pathlib import Path
from typing import Set, List, Dict

# Set random seed for reproducibility
RANDOM_SEED = 41
np.random.seed(RANDOM_SEED)

## 2. Load Dataset

In [186]:
# Load the dataset
data_path = Path('../../data/all_recipes_final.csv')
df = pd.read_csv(data_path)

print(f"Dataset loaded: {len(df)} recipes")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nMissing values:\n{df.isnull().sum()}")

Dataset loaded: 10263 recipes

Columns: ['title', 'type_of_food', 'link', 'description', 'ingredients', 'step', 'note', 'num_of_ingredients', 'cook_time', 'num_of_people', 'calories', 'source']

Missing values:
title                    0
type_of_food             0
link                     0
description              7
ingredients              0
step                     0
note                     0
num_of_ingredients       0
cook_time              295
num_of_people          294
calories              9826
source                   0
dtype: int64


In [187]:
# Handle missing data
# Drop rows with missing ingredients or type_of_food
df_clean = df.dropna(subset=['ingredients', 'type_of_food']).copy()

# Fill missing titles with placeholder
if 'title' in df_clean.columns:
    df_clean['title'] = df_clean['title'].fillna('Untitled Recipe')

# Reset index and create a unique ID
df_clean = df_clean.reset_index(drop=True)
df_clean['recipe_id'] = df_clean.index

print(f"After cleaning: {len(df_clean)} recipes")
print(f"Dropped {len(df) - len(df_clean)} recipes with missing critical data")

After cleaning: 10263 recipes
Dropped 0 recipes with missing critical data


## 3. Normalize Ingredients

### 3.1. Analyze Quantity Patterns in Ingredients

In [188]:
# Sample ingredients to identify quantity patterns
for idx in range(min(10, len(df_clean))):
    ingredients = df_clean.iloc[idx]['ingredients']
    if pd.notna(ingredients):
        # Split and show first few items
        items = [item.strip() for item in re.split(r'[,;\n]', ingredients) if item.strip()]
        for i, item in enumerate(items[:3]):
            print(f"  {item}")
        if idx < 9:
            print()

# Common quantity patterns in Vietnamese
quantity_patterns = [
    r'\d+\s*(gram|g|kg|kilogram)',  # 100 gram, 500g, 1kg
    r'\d+\s*(ml|lít|liter)',  # 200ml, 1 lít
    r'\d+\s*(muỗng|thìa)',  # 2 muỗng, 3 thìa
    r'\d+\s*con',  # 1 con, 2 con
    r'\d+\s*củ',  # 3 củ
    r'\d+\s*quả',  # 2 quả
    r'\d+\s*(cái|chiếc)',  # 5 cái
    r'nửa\s*\w+',  # nửa con, nửa kg
    r'một\s*ít',  # một ít
    r'\d+/\d+',  # 1/2, 1/4
    r'\d+',  # Any standalone numbers
]

  ['1 kg hành củ tươi'
  'Tro bếp hoặc nước vo gọa'
  'Muối hạt

  ['2 củ su hào non'
  '1 con mực khô'
  '1/2 củ cà rốt'

  ['800 gr măng khô'
  '2 móng giò lợn'
  'Nước dùng (gà hoặc ninh xương lợn)'

  ['2 bộ lòng mề gà'
  '100 gr lạc'
  '50 gr hạt đậu Hà Lan'

  ['500 gr giò sống'
  '300 gr bì lợn'
  '20 - 30 gr ớt xiêm xanh'

  ['300 gr giò sống'
  '200 gr nạc vai'
  '200 gr bì thăn'

  ['500 gr thịt chân giò'
  '4 cây sả'
  '4 tép tỏi'

  ['500 gr thịt nạc vai'
  '200 gr giá đỗ'
  '2 lòng đỏ trứng gà'

  ['1
  5 kg thịt ba chỉ liền khối'
  '5 củ hành khô'

  ['1 kg thịt ba rọi (ba chỉ rút sườn)'
  '10 quả trứng vịt'
  'Nước dừa tươi'


### 3.2. Remove Quantities from Ingredients

In [189]:
def remove_quantities(text: str) -> str:
    """
    "100g thịt bò" -> "thịt bò", "1 con gà" -> "gà"
    """
    if pd.isna(text) or not isinstance(text, str):
        return ""
    
    # Convert to lowercase first
    text = text.lower()
    
    # Remove ALL quotes, brackets, and parentheses
    text = text.replace("'", "").replace('"', "").replace('[', '').replace(']', '')
    text = text.replace('(', '').replace(')', '')
    
    # Define quantity patterns (Vietnamese specific)
    patterns_to_remove = [
        # Fractions FIRST (before other number patterns) - remove entire fraction
        r'\d+/\d+',
        
        # Measurements with units (including "cà phê" for teaspoon)
        r'\d+[\.,]?\d*\s*(gram|gr|g|kg|kilogram|kí|ký)\s*',
        r'\d+[\.,]?\d*\s*(ml|mililít|lít|liter|l)\s*',
        r'\d+[\.,]?\d*\s*(muỗng|thìa)\s*(cà\s*phê|canh)?\s*',
        r'\d+[\.,]?\d*\s*(mcf|mct)\s*',
        r'\d+[\.,]?\d*\s*(chén|bát|tô|ly|cốc)\s*',
        
        # Counters
        r'\d+[\.,]?\d*\s*(con|cái|chiếc|củ|quả|trái|trứng|lát|miếng|khúc|đốt|nhánh|cây|bông|tép|nắm)\s*',
        
        # Vietnamese number words + quantity
        r'(một|hai|ba|bốn|năm|sáu|bảy|tám|chín|mười)\s+(ít|chút|tí|vài)\s*',
        r'\d+\s+(ít|chút|tí|vài)\s*',
        
        # Descriptors
        r'nửa\s+\w*\s*',
        r'một\s+(ít|chút|tí)\s*',
        r'vài\s+\w*\s*',
        
        # Standalone numbers with spaces
        r'^\d+[\.,]?\d*\s+',
        r'\s+\d+[\.,]?\d*\s+',
        r'\s+\d+[\.,]?\d*$',
    ]
    
    # Apply all patterns
    for pattern in patterns_to_remove:
        text = re.sub(pattern, ' ', text, flags=re.IGNORECASE)
    
    # Clean up extra whitespace and trim
    text = re.sub(r'\s+', ' ', text).strip()
    
    # Remove leading/trailing special characters (but parentheses already removed)
    text = re.sub(r'^[^\w\sáàảãạăắằẳẵặâấầẩẫậéèẻẽẹêếềểễệíìỉĩịóòỏõọôốồổỗộơớờởỡợúùủũụưứừửữựýỳỷỹỵđ]+', '', text)
    text = re.sub(r'[^\w\sáàảãạăắằẳẵặâấầẩẫậéèẻẽẹêếềểễệíìỉĩịóòỏõọôốồổỗộơớờởỡợúùủũụưứừửữựýỳỷỹỵđ]+$', '', text)
    
    return text.strip()

In [190]:
# Test the function with problematic cases
test_cases = [
    "100 gram bắp non",
    "' r bắp non'",
    "'1 ít gia vị thông dụng (đường/ muối/ hạt nêm/ tiêu xay)'",
    "'1 ít hành ngò'",
    "'100gr bơ lạt'",
    "2 muỗng cà phê đường",
    "3 muỗng canh nước mắm",
    "1 muỗng tiêu",
    "nửa kg cà chua",
    "500ml nước mắm",
    "3 củ hành tây",
    "10gr thịt bò",
    "2 nhánh hành lá",
    "1 bông cải xanh",
    "3 tép tỏi",
    "1 nắm rau muống",
    "200g thịt băm",
    "1/2 quả chanh",
    "1/4 chén nước",
    "vài lát gừng",
    "bakingsoda (bột nở)",
    "dầu ăn (hoặc bơ)",
]

for test in test_cases:
    cleaned = remove_quantities(test)
    print(f"  Original: {test:55} -> Cleaned: '{cleaned}'")

  Original: 100 gram bắp non                                        -> Cleaned: 'bắp non'
  Original: ' r bắp non'                                            -> Cleaned: 'r bắp non'
  Original: '1 ít gia vị thông dụng (đường/ muối/ hạt nêm/ tiêu xay)' -> Cleaned: 'gia vị thông dụng đường/ muối/ hạt nêm/ tiêu xay'
  Original: '1 ít hành ngò'                                         -> Cleaned: 'hành ngò'
  Original: '100gr bơ lạt'                                          -> Cleaned: 'bơ lạt'
  Original: 2 muỗng cà phê đường                                    -> Cleaned: 'đường'
  Original: 3 muỗng canh nước mắm                                   -> Cleaned: 'nước mắm'
  Original: 1 muỗng tiêu                                            -> Cleaned: 'tiêu'
  Original: nửa kg cà chua                                          -> Cleaned: 'cà chua'
  Original: 500ml nước mắm                                          -> Cleaned: 'nước mắm'
  Original: 3 củ hành tây                                 

In [191]:
# normalize function
def normalize_ingredients(ingredients_str: str) -> Set[str]:
    if pd.isna(ingredients_str) or not isinstance(ingredients_str, str):
        return set()
    
    # Split first by separators
    normalized = re.sub(r'[,;\n]+', '|', ingredients_str)
    tokens = normalized.split('|')
    
    # Remove quantities from each token
    cleaned_tokens = []
    for token in tokens:
        cleaned = remove_quantities(token)
        if cleaned and len(cleaned) > 1:  # Only keep non-empty tokens with more than 1 char
            cleaned_tokens.append(cleaned)
    
    return set(cleaned_tokens)

### 3.3. Apply Normalization

In [192]:
# Apply the improved normalization function
df_clean['ingredients_normalized'] = df_clean['ingredients'].apply(normalize_ingredients)

for idx in range(min(3, len(df_clean))):
    original = df_clean.iloc[idx]['ingredients']
    norm = df_clean.iloc[idx]['ingredients_normalized']
    
    print(f"Recipe {idx}:")
    print(f"  Original: {original}")
    print(f"  Normalized: {list(norm)}\n")

Recipe 0:
  Original: ['1 kg hành củ tươi', 'Tro bếp hoặc nước vo gọa', 'Muối hạt, đường', 'Cà rốt trang trí (tùy chọn)', 'Lọ sạch']
  Normalized: ['muối hạt', 'lọ sạch', 'hành củ tươi', 'đường', 'tro bếp hoặc nước vo gọa', 'cà rốt trang trí tùy chọn']

Recipe 1:
  Original: ['2 củ su hào non', '1 con mực khô', '1/2 củ cà rốt', 'Gia vị: Mắm, muối, đường, hạt tiêu, rượu trắng, gừng', 'Rau mùi trang trí', 'Mỡ lợn hoặc dầu ăn']
  Normalized: ['rau mùi trang trí', 'su hào non', 'rượu trắng', 'muối', 'mỡ lợn hoặc dầu ăn', 'hạt tiêu', 'mực khô', 'gừng', 'đường', 'gia vị: mắm', 'củ cà rốt']

Recipe 2:
  Original: ['800 gr măng khô', '2 móng giò lợn', 'Nước dùng (gà hoặc ninh xương lợn)', 'Hành khô, hành củ', 'Gia vị: Nước mắm truyền thống, muối', 'Nước vo gạo ngâm măng']
  Normalized: ['gia vị: nước mắm truyền thống', 'măng khô', 'muối', 'hành khô', 'nước vo gạo ngâm măng', 'nước dùng gà hoặc ninh xương lợn', 'hành củ', 'móng giò lợn']



### 3.4 Export evaluation dataset

In [193]:
# Export with normalized ingredients as string
df_clean.to_csv("eval_dataset_cleaned.csv", index=False)
print("Evaluation dataset exported to 'eval_dataset_cleaned.csv'")

Evaluation dataset exported to 'eval_dataset_cleaned.csv'


## 4. Select Random Queries

In [194]:
# Configuration
NUM_QUERIES = 200
TOP_K = 10

# Select random queries
query_indices = np.random.choice(df_clean.index, size=NUM_QUERIES, replace=False)

queries_df = df_clean.loc[query_indices].copy()
print(f"Selected {len(queries_df)} queries")
print(f"\nType of food distribution in queries:")
print(queries_df['type_of_food'].value_counts())

Selected 200 queries

Type of food distribution in queries:
type_of_food
Món bánh                  40
Món chiên                 23
Món kho                   16
Ăn vặt                    13
Món xào                   13
Thức uống                 10
Món ngon hàng ngày         9
Món canh                   8
Món chay                   7
Món gỏi - salad            7
Món hấp                    6
Món nướng                  6
Món tráng miệng            4
Món chính                  4
Món cuốn - trộn            4
Món nước                   4
Món cháo                   3
Món kem                    3
Món từ bò                  3
Món khô - mắm              3
Bữa sáng đơn giản          2
Trà sữa                    2
Ngày lễ Tết                2
Món chè                    2
Món Tết                    1
Món ngon ngày lạnh         1
Món từ gà                  1
Món lẩu                    1
Món ngon cho cuối tuần     1
Quà - Món ăn vặt           1
Name: count, dtype: int64


## 5. Calculate Jaccard Similarity and Assign Relevance Grades

In [195]:
def jaccard_similarity(set1: Set[str], set2: Set[str]) -> float:
    if len(set1) == 0 and len(set2) == 0:
        return 0.0
    intersection = len(set1 & set2) # số nguyên liệu trùng (giao)
    union = len(set1 | set2)        # tổng số nguyên liệu (hợp)
    return intersection / union if union > 0 else 0.0

def assign_relevance_grade(jaccard_score: float) -> int:
    """
    Assign graded relevance based on Jaccard similarity.
    - < 0.10: 0 (not relevant)
    - 0.10 - 0.19: 1 (somewhat relevant)
    - 0.20 - 0.29: 2 (relevant)
    - >= 0.30: 3 (highly relevant)
    """
    if jaccard_score < 0.10:
        return 0
    elif jaccard_score < 0.20:
        return 1
    elif jaccard_score < 0.30:
        return 2
    else:
        return 3

In [196]:
test_set1 = {'a', 'b', 'c', 'd'}
test_set2 = {'b', 'c', 'e', 'f'}
test_jaccard = jaccard_similarity(test_set1, test_set2)
print(f"Test Jaccard: {test_jaccard:.3f}")
print(f"Test Relevance Grade: {assign_relevance_grade(test_jaccard)}")

Test Jaccard: 0.333
Test Relevance Grade: 3


## 6. Build Evaluation Data for All Queries

In [197]:
def find_top_k_similar(query_row, candidates_df, k=10):
    """
    Find top K most similar items to the query.
    Only considers candidates with the same type_of_food.
    Excludes the query itself from candidates.
    """
    query_id = query_row['recipe_id']
    query_type = query_row['type_of_food']
    query_ingredients = query_row['ingredients_normalized']
    
    # Filter candidates: same type, exclude query itself
    candidates = candidates_df[
        (candidates_df['type_of_food'] == query_type) &    # phải cùng type
        (candidates_df['recipe_id'] != query_id)            # tránh so sánh với chính nó
    ].copy()
    
    if len(candidates) == 0:
        return []
    
    # Calculate Jaccard similarity for all candidates
    candidates['jaccard'] = candidates['ingredients_normalized'].apply(
        lambda x: jaccard_similarity(query_ingredients, x)
    )
    
    # Sort by Jaccard and take top K
    top_k = candidates.nlargest(k, 'jaccard')
    
    # Prepare results
    results = []
    for _, row in top_k.iterrows():
        jaccard_score = row['jaccard']
        results.append({
            'doc_id': int(row['recipe_id']),
            'doc_title': row['title'] if 'title' in row else 'Untitled',
            'jaccard': float(jaccard_score),
            'rel': assign_relevance_grade(jaccard_score)
        })
    
    return results

In [198]:
# Process all queries (with progress indicator)
eval_data = []

for idx, (_, query_row) in enumerate(queries_df.iterrows()):
    if (idx + 1) % 50 == 0:
        print(f"  Processed {idx + 1}/{len(queries_df)} queries...")
    
    top_k_results = find_top_k_similar(query_row, df_clean, k=TOP_K)
    
    eval_entry = {
        'query_id': int(query_row['recipe_id']),
        'query_title': query_row['title'] if 'title' in query_row else 'Untitled',
        'query_type': query_row['type_of_food'],
        'query_ingredients': list(query_row['ingredients_normalized']),
        'top10': top_k_results
    }
    
    eval_data.append(eval_entry)

  Processed 50/200 queries...
  Processed 100/200 queries...
  Processed 150/200 queries...
  Processed 200/200 queries...


## 7. Inspect Sample Results

In [199]:
# Print first 2 queries as examples
for i in range(min(2, len(eval_data))):
    query = eval_data[i]
    print(f"\n{'='*80}")
    print(f"QUERY {i+1}")
    print(f"{'='*80}")
    print(f"Query ID: {query['query_id']}")
    print(f"Query Title: {query['query_title']}")
    print(f"Query Type: {query['query_type']}")
    print(f"Query Ingredients ({len(query['query_ingredients'])}): {query['query_ingredients']}")
    print(f"\nTop 10 Similar Items:")
    print(f"{'Rank':<6} {'Doc ID':<10} {'Jaccard':<10} {'Rel':<6} {'Title'}")
    print("-" * 80)
    
    for rank, item in enumerate(query['top10'], 1):
        title_short = item['doc_title']
        print(f"{rank:<6} {item['doc_id']:<10} {item['jaccard']:<10.3f} {item['rel']:<6} {title_short}")


QUERY 1
Query ID: 6302
Query Title: Cá rô kho gừng thơm ngon đậm đà hương vị cho bữa cơm
Query Type: Món kho
Query Ingredients (7): ['dầu ăn', 'gia vị thông dụng muối/ đường/ tiêu/ hạt nêm/ bột ngọt', 'ớt hiểm', 'nước mắm', 'nước màu điều', 'gừng', 'cá rô']

Top 10 Similar Items:
Rank   Doc ID     Jaccard    Rel    Title
--------------------------------------------------------------------------------
1      6305       0.333      3      Cá thu kho gừng cay cay ngon miệng đậm đà dễ làm
2      6264       0.308      3      Cá rô kho tộ thơm ngon đậm đà hấp dẫn cực đưa cơm
3      6530       0.300      3      Món cá trê kho tiêu thơm ngon khó cưỡng cực đưa cơm
4      6086       0.273      2      Thịt kho nghệ thơm ngon đậm đà dễ làm cho bữa cơm
5      6319       0.273      2      Cá ngát kho gừng thơm ngon, đậm đà cho bữa cơm thêm tròn vị
6      6290       0.250      2      Món cá diếc kho tiêu thơm ngon đậm đà hấp dẫn tại nhà
7      6446       0.250      2      Thịt kho củ sắn (củ đậu) đậm

## 8. Export Evaluation Data to JSONL

In [200]:
# Export to JSONL file
output_file = Path('eval_ground_truth.jsonl')

with open(output_file, 'w', encoding='utf-8') as f:
    for entry in eval_data:
        f.write(json.dumps(entry, ensure_ascii=False) + '\n')

print(f"Evaluation data exported to: {output_file}")
print(f"Total entries: {len(eval_data)}")

Evaluation data exported to: eval_ground_truth.jsonl
Total entries: 200


## 9. Summary Statistics

In [201]:
# Calculate statistics
all_jaccard_scores = []
all_relevance_grades = []

for query in eval_data:
    for item in query['top10']:
        all_jaccard_scores.append(item['jaccard'])
        all_relevance_grades.append(item['rel'])

print(f"\nTotal queries: {len(eval_data)}")
print(f"Total relevance judgments: {len(all_jaccard_scores)}")
print(f"\nJaccard Similarity Statistics:")
print(f"  Mean: {np.mean(all_jaccard_scores):.3f}")
print(f"  Median: {np.median(all_jaccard_scores):.3f}")
print(f"  Std: {np.std(all_jaccard_scores):.3f}")
print(f"  Min: {np.min(all_jaccard_scores):.3f}")
print(f"  Max: {np.max(all_jaccard_scores):.3f}")

print(f"\nRelevance Grade Distribution:")
grade_counts = pd.Series(all_relevance_grades).value_counts().sort_index()
for grade, count in grade_counts.items():
    percentage = count / len(all_relevance_grades) * 100
    print(f"  Grade {grade}: {count:4d} ({percentage:5.1f}%)")


Total queries: 200
Total relevance judgments: 1996

Jaccard Similarity Statistics:
  Mean: 0.201
  Median: 0.200
  Std: 0.112
  Min: 0.000
  Max: 1.000

Relevance Grade Distribution:
  Grade 0:  296 ( 14.8%)
  Grade 1:  689 ( 34.5%)
  Grade 2:  659 ( 33.0%)
  Grade 3:  352 ( 17.6%)
